# Task 3 - KMeans Clustering

Dans ce notebook, nous allons implémenter l'algorithme de clustering KMeans, un algorithme fondamental en machine learning non supervisé pour la découverte de groupes dans les données.

In [ ]:
# Importation des bibliothèques nécessaires
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

# Configuration de l'affichage
%matplotlib inline
plt.style.use('seaborn')

## 1. Génération de données synthétiques

Pour cette démonstration, nous allons générer un jeu de données synthétique avec des clusters bien définis.

In [ ]:
# Génération de données synthétiques avec 4 clusters
X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.60, random_state=0)

# Création d'un DataFrame pandas
df = pd.DataFrame(X, columns=['Feature_1', 'Feature_2'])

# Visualisation des données générées
plt.figure(figsize=(10, 8))
plt.scatter(X[:, 0], X[:, 1], s=50)
plt.title('Données synthétiques avec clusters')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Shape of dataset: {df.shape}")
df.head()

In [ ]:
# Exploration des données
print("Statistiques descriptives:")
df.describe()

## 2. Prétraitement des données

Normalisation des données pour assurer une égalité de traitement entre les caractéristiques.

In [ ]:
# Normalisation des données
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convertir en DataFrame pour faciliter la manipulation
X_scaled_df = pd.DataFrame(X_scaled, columns=['Feature_1', 'Feature_2'])
print("Données après normalisation:")
X_scaled_df.head()

## 3. Détermination du nombre optimal de clusters

Utilisation de la méthode du coude pour déterminer le nombre optimal de clusters.

In [ ]:
# Méthode du coude pour déterminer le nombre optimal de clusters
inertias = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

# Visualisation de la méthode du coude
plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, marker='o')
plt.title('Méthode du coude pour la détermination du nombre optimal de clusters')
plt.xlabel('Nombre de clusters (k)')
plt.ylabel('Inertie (Within-Cluster Sum of Squares)')
plt.grid(True, alpha=0.3)
plt.show()

## 4. Application de KMeans avec le nombre optimal de clusters

Entraînement du modèle KMeans avec le nombre de clusters déterminé.

In [ ]:
# Application de KMeans avec k=4 (nombre optimal identifié)
k_optimal = 4
kmeans = KMeans(n_clusters=k_optimal, random_state=42)
y_pred = kmeans.fit_predict(X_scaled)

# Centroides des clusters
centroids = kmeans.cluster_centers_
centroids_original = scaler.inverse_transform(centroids)

print(f"Nombre de clusters: {k_optimal}")
print(f"Inertie: {kmeans.inertia_:.2f}")
print(f"Centroides (données normalisées):\n{centroids}")

## 5. Visualisation des clusters

Visualisation des clusters obtenus et des centroides.

In [ ]:
# Visualisation des clusters
plt.figure(figsize=(12, 8))

# Plot des points de données colorés par cluster
scatter = plt.scatter(X[:, 0], X[:, 1], c=y_pred, cmap='viridis', s=50, alpha=0.7)

# Plot des centroides
plt.scatter(centroids_original[:, 0], centroids_original[:, 1], 
            c='red', marker='x', s=200, linewidths=3, label='Centroides')

plt.title('Clusters obtenus avec KMeans')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.colorbar(scatter)
plt.show()

## 6. Évaluation de la qualité du clustering

Utilisation du coefficient de silhouette pour évaluer la qualité du clustering.

In [ ]:
# Calcul du coefficient de silhouette
silhouette_avg = silhouette_score(X_scaled, y_pred)
print(f"Coefficient de silhouette moyen: {silhouette_avg:.3f}")

# Calcul du coefficient de silhouette pour chaque cluster
from sklearn.metrics import silhouette_samples
silhouette_vals = silhouette_samples(X_scaled, y_pred)

# Visualisation du coefficient de silhouette pour chaque cluster
plt.figure(figsize=(10, 6))
y_lower = 10

for i in range(k_optimal):
    # Agrégation des coefficients de silhouette pour le cluster i
    cluster_silhouette_vals = silhouette_vals[y_pred == i]
    cluster_silhouette_vals.sort()
    
    size_cluster_i = cluster_silhouette_vals.shape[0]
    y_upper = y_lower + size_cluster_i
    
    plt.fill_betweenx(np.arange(y_lower, y_upper),
                      0, cluster_silhouette_vals,
                      alpha=0.7)
    
    # Étiquette du cluster au milieu de ses valeurs
    plt.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
    
    # Calcul de la nouvelle valeur de y_lower pour le prochain tracé
    y_lower = y_upper + 10

plt.axvline(x=silhouette_avg, color="red", linestyle="--", 
            label=f'Moyenne: {silhouette_avg:.3f}')
plt.xlabel('Coefficient de silhouette')
plt.ylabel('Cluster')
plt.title('Coefficient de silhouette pour chaque cluster')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7. Comparaison avec différents nombres de clusters

Comparaison de la qualité du clustering pour différents nombres de clusters.

In [ ]:
# Comparaison de la qualité du clustering pour différents nombres de clusters
K_range = range(2, 11)
silhouette_scores = []
inertias = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    cluster_labels = kmeans.fit_predict(X_scaled)
    
    # Calcul du coefficient de silhouette
    silhouette_avg = silhouette_score(X_scaled, cluster_labels)
    silhouette_scores.append(silhouette_avg)
    
    # Stockage de l'inertie
    inertias.append(kmeans.inertia_)

# Visualisation de la comparaison
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Graphique de l'inertie
axes[0].plot(K_range, inertias, marker='o')
axes[0].set_title('Méthode du coude')
axes[0].set_xlabel('Nombre de clusters (k)')
axes[0].set_ylabel('Inertie')
axes[0].grid(True, alpha=0.3)

# Graphique du coefficient de silhouette
axes[1].plot(K_range, silhouette_scores, marker='o')
axes[1].set_title('Coefficient de silhouette')
axes[1].set_xlabel('Nombre de clusters (k)')
axes[1].set_ylabel('Coefficient de silhouette')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Meilleur score de silhouette
best_k = K_range[np.argmax(silhouette_scores)]
best_silhouette = max(silhouette_scores)
print(f"Meilleur nombre de clusters: {best_k}")
print(f"Meilleur coefficient de silhouette: {best_silhouette:.3f}")

## 8. Analyse des caractéristiques des clusters

Analyse des propriétés moyennes des points dans chaque cluster.

In [ ]:
# Ajout des labels de cluster au DataFrame
df['Cluster'] = y_pred

# Statistiques descriptives par cluster
cluster_stats = df.groupby('Cluster').agg({
    'Feature_1': ['mean', 'std'],
    'Feature_2': ['mean', 'std']
}).round(2)

print("Statistiques descriptives par cluster:")
print(cluster_stats)

# Visualisation des statistiques par cluster
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Moyennes par cluster
cluster_means = df.groupby('Cluster')[['Feature_1', 'Feature_2']].mean()
cluster_means.plot(kind='bar', ax=axes[0])
axes[0].set_title('Moyennes des caractéristiques par cluster')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Valeur moyenne')
axes[0].legend(['Feature 1', 'Feature 2'])
axes[0].grid(True, alpha=0.3)

# Écarts-types par cluster
cluster_stds = df.groupby('Cluster')[['Feature_1', 'Feature_2']].std()
cluster_stds.plot(kind='bar', ax=axes[1])
axes[1].set_title('Écarts-types des caractéristiques par cluster')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Écart-type')
axes[1].legend(['Feature 1', 'Feature 2'])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Exemple avec un jeu de données réel

Application de KMeans sur un jeu de données réel : le jeu de données Iris.

In [ ]:
# Chargement du jeu de données Iris
from sklearn.datasets import load_iris

iris = load_iris()
X_iris = iris.data
y_iris_true = iris.target
feature_names = iris.feature_names

# Création d'un DataFrame
df_iris = pd.DataFrame(X_iris, columns=feature_names)
print("Jeu de données Iris:")
print(df_iris.head())

# Normalisation des données
scaler_iris = StandardScaler()
X_iris_scaled = scaler_iris.fit_transform(X_iris)

# Application de KMeans avec k=3
kmeans_iris = KMeans(n_clusters=3, random_state=42)
y_iris_pred = kmeans_iris.fit_predict(X_iris_scaled)

# Évaluation
silhouette_iris = silhouette_score(X_iris_scaled, y_iris_pred)
print(f"\nCoefficient de silhouette pour Iris: {silhouette_iris:.3f}")

# Visualisation (utilisation des deux premières caractéristiques)
plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris_pred, cmap='viridis', s=50, alpha=0.7)
plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.title('Clustering KMeans sur le jeu de données Iris')
plt.colorbar(scatter)
plt.grid(True, alpha=0.3)
plt.show()

## Conclusion

Dans ce notebook, nous avons implémenté l'algorithme de clustering KMeans :
- Génération et exploration de données synthétiques
- Prétraitement des données (normalisation)
- Détermination du nombre optimal de clusters avec la méthode du coude
- Application de l'algorithme KMeans
- Visualisation des clusters et des centroides
- Évaluation de la qualité du clustering avec le coefficient de silhouette
- Comparaison de la qualité pour différents nombres de clusters
- Analyse des caractéristiques des clusters
- Application sur un jeu de données réel (Iris)

Le clustering KMeans est un algorithme simple mais puissant pour la découverte de structures dans les données. Il est important de normaliser les données avant l'application de l'algorithme et de choisir judicieusement le nombre de clusters avec des méthodes comme celle du coude ou le coefficient de silhouette.